# Ground Tracks, Reference Frames, and TLE Interpretation

This notebook builds intuition for how orbital motion appears in Earth-fixed coordinates, and why simple two-body propagation differs from operational TLE/SGP4 predictions.

We will:

1. Build a **two-body Keplerian propagator** in ECI.
2. Rotate ECI to ECEF using Earth's rotation rate.
3. Convert ECEF vectors to latitude/longitude and plot ground tracks.
4. Load a real ISS TLE and propagate with **SGP4**.
5. Compare two-body vs SGP4 results and explain differences.
6. Contrast polar, equatorial, and sun-synchronous style ground tracks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sgp4.api import Satrec, jday

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True

## 1) Constants and helper functions

We assume Earth rotates at:

\[
\omega_E = 7.2921159\times10^{-5}\;\text{rad/s}
\]

We use a simple ECI \rightarrow ECEF rotation about the inertial \(z\)-axis by angle \(\theta = \omega_E t\) (ignoring precession, nutation, polar motion, and UT1 corrections).

In [ ]:
mu = 398600.4418
omega_E = 7.2921159e-5
R_E = 6378.137

def rot3(angle_rad):
    c = np.cos(angle_rad)
    s = np.sin(angle_rad)
    return np.array([[c, -s, 0.0],
                     [s,  c, 0.0],
                     [0.0, 0.0, 1.0]])

def eci_to_ecef(r_eci, t_sec, theta0=0.0):
    theta = theta0 + omega_E * t_sec
    return rot3(theta) @ r_eci

def ecef_to_latlon(r_ecef):
    x, y, z = r_ecef
    lon = np.degrees(np.arctan2(y, x))
    rho = np.sqrt(x*x + y*y)
    lat = np.degrees(np.arctan2(z, rho))
    return lat, lon

def wrap_lon(lon_deg):
    return (lon_deg + 180.0) % 360.0 - 180.0

## 2) Keplerian state generation and two-body propagation (24 hours)

Initial orbit (classical elements):

- Semi-major axis: \(a = 7000\) km
- Eccentricity: \(e = 0.01\)
- Inclination: \(i = 98^\circ\)
- (Chosen for demonstration) \(\Omega = 40^\circ, \omega = 30^\circ, 
u_0 = 0^\circ\)

For a pure two-body model, these elements remain constant and mean anomaly evolves linearly.

In [ ]:
def kepler_E_from_M(M, e, tol=1e-12, max_iter=50):
    M = np.mod(M, 2*np.pi)
    E = M if e < 0.8 else np.pi
    for _ in range(max_iter):
        f = E - e*np.sin(E) - M
        fp = 1 - e*np.cos(E)
        dE = -f/fp
        E = E + dE
        if abs(dE) < tol:
            break
    return E

def perifocal_to_eci_matrix(raan, inc, argp):
    cO, sO = np.cos(raan), np.sin(raan)
    ci, si = np.cos(inc), np.sin(inc)
    cw, sw = np.cos(argp), np.sin(argp)

    return np.array([
        [cO*cw - sO*sw*ci, -cO*sw - sO*cw*ci,  sO*si],
        [sO*cw + cO*sw*ci, -sO*sw + cO*cw*ci, -cO*si],
        [sw*si,              cw*si,             ci]
    ])

def kepler_state_eci(a, e, inc, raan, argp, nu):
    p = a*(1 - e**2)
    r_pf = np.array([
        p*np.cos(nu)/(1 + e*np.cos(nu)),
        p*np.sin(nu)/(1 + e*np.cos(nu)),
        0.0
    ])
    v_pf = np.array([
        -np.sqrt(mu/p)*np.sin(nu),
        np.sqrt(mu/p)*(e + np.cos(nu)),
        0.0
    ])
    Q = perifocal_to_eci_matrix(raan, inc, argp)
    return Q @ r_pf, Q @ v_pf

def propagate_two_body_kepler(a, e, inc_deg, raan_deg, argp_deg, nu0_deg, t_array):
    inc = np.radians(inc_deg)
    raan = np.radians(raan_deg)
    argp = np.radians(argp_deg)
    nu0 = np.radians(nu0_deg)

    E0 = 2*np.arctan2(np.sqrt(1-e)*np.sin(nu0/2), np.sqrt(1+e)*np.cos(nu0/2))
    M0 = E0 - e*np.sin(E0)
    n = np.sqrt(mu/a**3)

    rs, vs = [], []
    for t in t_array:
        M = M0 + n*t
        E = kepler_E_from_M(M, e)
        nu = 2*np.arctan2(np.sqrt(1+e)*np.sin(E/2), np.sqrt(1-e)*np.cos(E/2))
        r_eci, v_eci = kepler_state_eci(a, e, inc, raan, argp, nu)
        rs.append(r_eci)
        vs.append(v_eci)
    return np.array(rs), np.array(vs)

a = 7000.0
e = 0.01
i = 98.0
raan = 40.0
argp = 30.0
nu0 = 0.0

t_24h = np.arange(0.0, 24*3600 + 60, 60.0)
r_eci_2b, v_eci_2b = propagate_two_body_kepler(a, e, i, raan, argp, nu0, t_24h)

lat_2b = np.zeros_like(t_24h)
lon_2b = np.zeros_like(t_24h)
for k, t in enumerate(t_24h):
    r_ecef = eci_to_ecef(r_eci_2b[k], t)
    lat_2b[k], lon_2b[k] = ecef_to_latlon(r_ecef)
lon_2b = wrap_lon(lon_2b)

print(f"Two-body samples: {len(t_24h)}")

## 3) Two-body ground track plot and nodal regression intuition

In a pure two-body inertial frame, the orbital plane is fixed. However, when viewed in Earth-fixed coordinates, the ascending node longitude appears to drift westward each orbit because Earth rotates eastward beneath the orbit.

This is often called **apparent nodal regression in the ground track** (distinct from true J2 nodal precession in inertial space).

In [ ]:
def plot_world_axes(ax, title):
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    ax.set_xlabel('Longitude [deg]')
    ax.set_ylabel('Latitude [deg]')
    ax.set_title(title)
    for lon in range(-180, 181, 30):
        ax.axvline(lon, color='lightgray', lw=0.5, zorder=0)
    for lat in range(-90, 91, 30):
        ax.axhline(lat, color='lightgray', lw=0.5, zorder=0)

asc_node_lons, asc_node_times = [], []
for k in range(1, len(lat_2b)):
    if lat_2b[k-1] < 0 and lat_2b[k] >= 0:
        frac = -lat_2b[k-1] / (lat_2b[k] - lat_2b[k-1] + 1e-15)
        lon_cross = lon_2b[k-1] + frac*(lon_2b[k] - lon_2b[k-1])
        lon_cross = wrap_lon(lon_cross)
        t_cross = t_24h[k-1] + frac*(t_24h[k] - t_24h[k-1])
        asc_node_lons.append(lon_cross)
        asc_node_times.append(t_cross/3600.0)

fig, ax = plt.subplots(1, 1)
plot_world_axes(ax, 'Two-body Ground Track (24 h)')
ax.plot(lon_2b, lat_2b, '.', ms=1.5, label='Two-body track')
if asc_node_lons:
    ax.plot(asc_node_lons, np.zeros(len(asc_node_lons)), 'ro', ms=4, label='Ascending nodes')
ax.legend(loc='upper right')
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(10,4))
ax.plot(asc_node_times, asc_node_lons, 'o-')
ax.set_xlabel('Time [hours]')
ax.set_ylabel('Ascending node longitude [deg]')
ax.set_title('Apparent westward shift of node longitudes in ECEF')
ax.grid(True)
plt.show()

## 4) Real ISS TLE + SGP4 propagation (3 days)

Below is a hardcoded ISS (ZARYA) example TLE. If you rerun this notebook in the future, consider replacing these lines with a newer TLE because TLE accuracy degrades with age.

In [ ]:
tle_line1 = "1 25544U 98067A   24060.54791435  .00016717  00000+0  30044-3 0  9993"
tle_line2 = "2 25544  51.6412 194.8652 0005939  20.2723 113.9682 15.50082721442169"

sat = Satrec.twoline2rv(tle_line1, tle_line2)

epoch_jd = sat.jdsatepoch + sat.jdsatepochF
bstar = sat.bstar
mean_motion_rad_min = sat.no_kozai
mean_motion_rev_day = mean_motion_rad_min * 1440.0 / (2*np.pi)
inclination_deg = np.degrees(sat.inclo)

print("TLE interpretation:")
print(f"  Epoch (Julian Date): {epoch_jd:.8f}")
print(f"  BSTAR drag term:     {bstar:.6e} [1/Earth radii]")
print(f"  Mean motion:         {mean_motion_rev_day:.8f} rev/day")
print(f"  Inclination:         {inclination_deg:.4f} deg")

def jd_to_calendar(jd):
    jd += 0.5
    Z = int(jd)
    F = jd - Z
    if Z < 2299161:
        A = Z
    else:
        alpha = int((Z - 1867216.25)/36524.25)
        A = Z + 1 + alpha - int(alpha/4)
    B = A + 1524
    C = int((B - 122.1)/365.25)
    D = int(365.25*C)
    E = int((B - D)/30.6001)
    day = B - D - int(30.6001*E) + F
    month = E - 1 if E < 14 else E - 13
    year = C - 4716 if month > 2 else C - 4715
    day_i = int(day)
    frac = day - day_i
    hours = int(frac*24)
    minutes = int((frac*24 - hours)*60)
    seconds = ((frac*24 - hours)*60 - minutes)*60
    return year, month, day_i, hours, minutes, seconds

yr, mo, dy, hh, mm, ss = jd_to_calendar(epoch_jd)
print(f"  Epoch (UTC approx):  {yr:04d}-{mo:02d}-{dy:02d} {hh:02d}:{mm:02d}:{ss:05.2f}")

minutes = np.arange(0.0, 3*24*60 + 2, 2.0)
lat_sgp4, lon_sgp4 = [], []

for m in minutes:
    jd = sat.jdsatepoch
    fr = sat.jdsatepochF + m/1440.0
    e_sgp4, r_km, v_kmps = sat.sgp4(jd, fr)
    if e_sgp4 != 0:
        lat_sgp4.append(np.nan)
        lon_sgp4.append(np.nan)
        continue
    t_sec = m * 60.0
    r_ecef = eci_to_ecef(np.array(r_km), t_sec)
    lat, lon = ecef_to_latlon(r_ecef)
    lat_sgp4.append(lat)
    lon_sgp4.append(wrap_lon(lon))

lat_sgp4 = np.array(lat_sgp4)
lon_sgp4 = np.array(lon_sgp4)
print(f"SGP4 samples: {len(minutes)}")

# Capture SGP4 state at TLE epoch to build a fair same-initial-state two-body comparison
err0, r0_epoch_km, v0_epoch_kmps = sat.sgp4(sat.jdsatepoch, sat.jdsatepochF)
if err0 != 0:
    raise RuntimeError(f"SGP4 error at epoch: {err0}")
r0_epoch_km = np.array(r0_epoch_km, dtype=float)
v0_epoch_kmps = np.array(v0_epoch_kmps, dtype=float)


## 5) Fair comparison: SGP4 vs two-body from the **same** initial state

A direct comparison should start from the same orbit at the same epoch. To do that:

- We take the ISS state vector \((\mathbf r_0, \mathbf v_0)\) from SGP4 at the TLE epoch.
- We propagate a pure two-body model from that exact initial state.
- We compare the resulting ground tracks.

This isolates **model physics differences** (e.g., drag/J2-like secular effects represented in SGP4) instead of mixing in different starting orbits.

In [ ]:
def two_body_acc(r_km):
    rnorm = np.linalg.norm(r_km)
    return -mu * r_km / (rnorm**3)


def rk4_two_body(r0, v0, dt, n_steps):
    r = np.array(r0, dtype=float)
    v = np.array(v0, dtype=float)
    rs = [r.copy()]
    vs = [v.copy()]

    for _ in range(n_steps):
        a1 = two_body_acc(r)
        k1_r = v
        k1_v = a1

        a2 = two_body_acc(r + 0.5*dt*k1_r)
        k2_r = v + 0.5*dt*k1_v
        k2_v = a2

        a3 = two_body_acc(r + 0.5*dt*k2_r)
        k3_r = v + 0.5*dt*k2_v
        k3_v = a3

        a4 = two_body_acc(r + dt*k3_r)
        k4_r = v + dt*k3_v
        k4_v = a4

        r = r + (dt/6.0)*(k1_r + 2*k2_r + 2*k3_r + k4_r)
        v = v + (dt/6.0)*(k1_v + 2*k2_v + 2*k3_v + k4_v)

        rs.append(r.copy())
        vs.append(v.copy())

    return np.array(rs), np.array(vs)


# Compare over 24 h at same 2-minute cadence as SGP4 subset
dt = 120.0
t_cmp = np.arange(0.0, 24*3600 + dt, dt)
rs_2b_same, vs_2b_same = rk4_two_body(r0_epoch_km, v0_epoch_kmps, dt, len(t_cmp)-1)

lat_2b_same, lon_2b_same = [], []
for k, tsec in enumerate(t_cmp):
    r_ecef = eci_to_ecef(rs_2b_same[k], tsec)
    la, lo = ecef_to_latlon(r_ecef)
    lat_2b_same.append(la)
    lon_2b_same.append(wrap_lon(lo))
lat_2b_same = np.array(lat_2b_same)
lon_2b_same = np.array(lon_2b_same)

# SGP4 first 24 h (already sampled every 2 min)
mask_24h = minutes <= 24*60
lat_sgp4_24 = lat_sgp4[mask_24h]
lon_sgp4_24 = lon_sgp4[mask_24h]

fig, ax = plt.subplots(1, 1)
plot_world_axes(ax, 'Fair comparison: same initial state at TLE epoch (first 24 h)')
ax.plot(lon_2b_same, lat_2b_same, '.', ms=1.8, label='Two-body from SGP4 epoch state')
ax.plot(lon_sgp4_24, lat_sgp4_24, '.', ms=1.8, label='SGP4 from TLE')
ax.legend(loc='upper right')
plt.show()

# Optional diagnostic: lat/lon differences point-by-point (approximate)
dlat = lat_sgp4_24 - lat_2b_same[:len(lat_sgp4_24)]
dlon = wrap_lon(lon_sgp4_24 - lon_2b_same[:len(lon_sgp4_24)])

fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].plot(t_cmp[:len(dlat)]/3600.0, dlat)
axes[0].set_xlabel('Time from TLE epoch [h]')
axes[0].set_ylabel('Latitude difference [deg] (SGP4 - 2body)')
axes[0].set_title('Latitude difference growth')
axes[0].grid(True)

axes[1].plot(t_cmp[:len(dlon)]/3600.0, dlon)
axes[1].set_xlabel('Time from TLE epoch [h]')
axes[1].set_ylabel('Longitude difference [deg] (SGP4 - 2body)')
axes[1].set_title('Longitude difference growth')
axes[1].grid(True)
plt.show()

# Keep the original illustrative 98-deg-vs-ISS comparison as a separate pedagogical overlay
fig, ax = plt.subplots(1, 1)
plot_world_axes(ax, 'Illustrative comparison: demo 98° Kepler orbit vs ISS SGP4 (not same initial orbit)')
ax.plot(lon_2b, lat_2b, '.', ms=1.2, label='Demo two-body (a=7000, e=0.01, i=98°)')
ax.plot(lon_sgp4_24, lat_sgp4_24, '.', ms=1.2, label='ISS SGP4 (24 h)')
ax.legend(loc='upper right')
plt.show()

### Why the fair-comparison tracks differ

With the same initial \((\mathbf r_0, \mathbf v_0)\), differences now primarily come from model physics and model formulation:

1. **Two-body model omissions**
   - No atmospheric drag.
   - No Earth oblateness perturbations (e.g., J2-driven secular effects).
   - No additional periodic/secular corrections used in TLE-compatible propagation.

2. **TLE + SGP4 is a coupled system**
   - TLE mean elements are estimated specifically for SGP4/SDP4.
   - SGP4 includes analytical terms (including drag through BSTAR) that produce trajectory evolution absent in pure Kepler propagation.

3. **Reference-frame simplification in this notebook**
   - We apply a simple constant-rate Earth rotation for ECI→ECEF to keep the notebook readable.
   - High-precision workflows would apply full time-scale and Earth-orientation handling.

## 6) Polar vs equatorial vs sun-synchronous-style tracks (two-body demo)

We keep \(a=7000\) km and \(e=0.01\), vary inclination:

- Equatorial: \(i=0^\circ\)
- Polar: \(i=90^\circ\)
- Sun-synchronous-like: \(i\approx98^\circ\)

Note: true Sun-synchronous behavior requires J2-driven RAAN precession matched to Earth's annual motion. A pure two-body model cannot reproduce true Sun-synch maintenance; it only shows the high-inclination geometry.

In [ ]:
configs = [
    (0.0, 'Equatorial (i=0°)'),
    (90.0, 'Polar (i=90°)'),
    (98.0, 'Sun-synchronous-like geometry (i=98°)')
]

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

for ax, (inc_deg, label) in zip(axes, configs):
    r_eci, _ = propagate_two_body_kepler(a, e, inc_deg, raan, argp, nu0, t_24h)
    lats, lons = [], []
    for k, t in enumerate(t_24h):
        r_ecef = eci_to_ecef(r_eci[k], t)
        la, lo = ecef_to_latlon(r_ecef)
        lats.append(la)
        lons.append(wrap_lon(lo))
    plot_world_axes(ax, label)
    ax.plot(lons, lats, '.', ms=1.3)

plt.tight_layout()
plt.show()

## 7) Why SGP4 is valid mainly in the LEO/TLE operational regime

SGP4 is the standard model for publicly distributed NORAD-style TLEs, especially for near-Earth satellites. Practical limitations:

1. **Model-data coupling**
   - TLEs are not osculating states; they are mean elements fitted for SGP4/SDP4 use.
   - Using TLEs with non-SGP4 propagators can introduce large errors.

2. **Best near epoch**
   - Accuracy is best near the TLE epoch and degrades with time.
   - Operationally, TLEs are refreshed frequently.

3. **Atmospheric drag uncertainty (LEO)**
   - Drag is strongly variable with space weather and density model mismatch.
   - BSTAR is an effective parameter, not a direct physical drag coefficient.

4. **Not a precision OD product**
   - For conjunction assessment, maneuvers, and precise operations, higher-fidelity force models + estimation are preferred.

5. **Regime assumptions**
   - SGP4 has specific analytical assumptions and deep-space/near-Earth branch behavior; it is not a universal high-precision propagator for all orbital regimes and time spans.

In practice: use SGP4 for **catalog-level prediction and screening**, and switch to higher-fidelity orbit determination products when mission/accuracy needs demand it.